# Module 4 Lab 1: pandas on a Real Dataset
**Daz Arnaud** | CS 79F Machine Learning on AWS

Portfolio repository URL: ____________________ (Daz: paste your GitHub repo URL here)

## Step 1: Load the file and profile it

In [1]:
import pandas as pd
print(pd.__version__)

df = pd.read_csv('m4_ecommerce_orders.csv')
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe())

3.0.5
(500, 8)
order_id              str
order_date            str
category              str
region                str
quantity            int64
unit_price        float64
shipping_cost     float64
payment_method        str
dtype: object
order_id           0
order_date         0
category           0
region             0
quantity           0
unit_price        21
shipping_cost      0
payment_method     0
dtype: int64
         quantity  unit_price  shipping_cost
count  500.000000  479.000000     500.000000
mean     5.088000  189.523591      12.179560
std      2.653741  115.619862       8.305565
min      1.000000    5.070000     -22.560000
25%      3.000000   87.805000       5.515000
50%      5.000000  177.790000      12.620000
75%      7.000000  293.770000      19.017500
max      9.000000  398.810000      24.980000


In [2]:
# Check the text columns too, since describe() only covers numbers
for col in ['category', 'region', 'payment_method']:
    print(df[col].value_counts(dropna=False), '\n')
print('Duplicate order_id values:', df['order_id'].duplicated().sum())
print('Fully duplicated rows:', df.duplicated().sum())

category
Home           103
Toys           102
Apparel        100
Electronics     99
Grocery         96
Name: count, dtype: int64 

region
South      130
West       130
East       123
Central    117
Name: count, dtype: int64 

payment_method
card      180
paypal    167
gift      153
Name: count, dtype: int64 

Duplicate order_id values: 0
Fully duplicated rows: 0


**Problem columns from the profile**

* `unit_price`: 21 values are missing (`isna()` shows 21, `describe()` count is 479 of 500).
* `shipping_cost`: the minimum is negative. A shipping cost cannot be below zero.
* `order_date`: loaded as text (`str`), not as a date, so I cannot sort or group by month without converting it.
* `quantity` looks fine: every value is between 1 and 9.

## Step 2: Sort the problems into two kinds

In [3]:
# Missing values
print(df.isna().sum())

# Impossible values, one check each
print((df['shipping_cost'] < 0).sum(), 'negative shipping costs')
print((df['quantity'] < 1).sum(), 'quantities below one')
# My own checks
print((df['unit_price'] <= 0).sum(), 'unit prices at or below zero')
print((df['shipping_cost'] == 0).sum(), 'shipping costs of exactly zero')
print(pd.to_datetime(df['order_date'], errors='coerce').isna().sum(), 'order dates that do not parse')

order_id           0
order_date         0
category           0
region             0
quantity           0
unit_price        21
shipping_cost      0
payment_method     0
dtype: int64
13 negative shipping costs
0 quantities below one
0 unit prices at or below zero
1 shipping costs of exactly zero
0 order dates that do not parse


**Missing**
* `unit_price`: 21 rows have no price.

**Impossible (present but cannot be true)**
* `shipping_cost` below 0: rule broken is "a shipping charge must be zero or more". 13 rows.
* `quantity` below 1: rule is "an order has at least one item". 0 rows, so this rule holds in this file.
* `unit_price` at or below 0: rule is "a product has a positive price". 0 rows.

**Type problem**
* `order_date` is stored as text. Every value parses as a date, so this is a type problem, not a bad value.

## Step 3: Fix two problems and say what each fix costs

In [4]:
# run this before you decide which fix to use
before = df['quantity'].mean()
after = df.dropna(subset=['unit_price'])['quantity'].mean()
print(before, after)
print('Change in mean quantity: {:.2f}%'.format((after - before) / before * 100))

# Do the other columns shift when the price rows are dropped?
for col in ['quantity', 'shipping_cost']:
    b = df[col].mean(); a = df.dropna(subset=['unit_price'])[col].mean()
    print(col, round(b, 3), round(a, 3), '{:+.2f}%'.format((a - b) / b * 100))
print(df[df['unit_price'].isna()]['category'].value_counts())

5.088 5.079331941544885
Change in mean quantity: -0.17%
quantity 5.088 5.079 -0.17%
shipping_cost 12.18 12.209 +0.24%
category
Apparel        7
Electronics    7
Home           3
Grocery        2
Toys           2
Name: count, dtype: int64


**Fix 1: negative shipping costs**

WHAT: 13 shipping_cost values are negative (as low as -22.56), and a shipping charge cannot be below zero.<br>
DID: turned those 13 values into missing (NaN) instead of flipping the sign or deleting the orders.<br>
COSTS: shipping averages are now based on 487 orders instead of 500, and if the negatives were really refunds or credits, I have thrown away that information.

**Fix 2: missing unit_price**

WHAT: 21 orders have no unit_price, so their order value cannot be calculated.<br>
DID: kept the rows, left the price as missing, and added a `price_missing` flag column instead of filling in a made-up price.<br>
COSTS: any revenue or order-value calculation has to skip these 21 orders, so revenue totals are understated for whatever categories they belong to.

**Fix 3: order_date stored as text**

WHAT: order_date loaded as text, so date sorting and monthly grouping do not work.<br>
DID: converted it with `pd.to_datetime`.<br>
COSTS: very little here because every value parsed, but if a future file has a bad date, `errors='coerce'` would silently turn it into a missing date.

In [5]:
import numpy as np
df_clean = df.copy()

# Fix 1: negative shipping costs become missing
neg = df_clean['shipping_cost'] < 0
df_clean.loc[neg, 'shipping_cost'] = np.nan

# Fix 2: keep missing prices but flag them
df_clean['price_missing'] = df_clean['unit_price'].isna()

# Fix 3: real dates
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])

print(df_clean.shape)
print(df_clean.dtypes)
print(df_clean.isna().sum())

(500, 9)
order_id                     str
order_date        datetime64[us]
category                     str
region                       str
quantity                   int64
unit_price               float64
shipping_cost            float64
payment_method               str
price_missing               bool
dtype: object
order_id           0
order_date         0
category           0
region             0
quantity           0
unit_price        21
shipping_cost     13
payment_method     0
price_missing      0
dtype: int64


## Step 4: Answer questions about the cleaned table

In [6]:
# Which category has the highest average quantity per order?
print(df_clean.groupby('category')['quantity'].mean().sort_values(ascending=False))

category
Toys           5.303922
Electronics    5.262626
Home           5.242718
Apparel        4.990000
Grocery        4.614583
Name: quantity, dtype: float64


In [7]:
# Which region has the highest average shipping cost?
print(df_clean.groupby('region')['shipping_cost'].mean().sort_values(ascending=False))

region
South      13.457969
West       12.900645
Central    12.808393
East       12.076585
Name: shipping_cost, dtype: float64


In [8]:
# Third aggregation: average order value (quantity x unit price) by category, priced orders only
df_clean['order_value'] = df_clean['quantity'] * df_clean['unit_price']
print(df_clean.groupby('category')['order_value'].mean().sort_values(ascending=False).round(2))

category
Toys           1065.95
Home           1037.13
Grocery         929.40
Electronics     918.43
Apparel         873.14
Name: order_value, dtype: float64


## Break it on purpose

In [9]:
# Break 1: mean shipping cost before and after removing the negative values
print('Before:', df['shipping_cost'].mean())
print('After :', df_clean['shipping_cost'].mean())

Before: 12.179559999999999
After : 12.817782340862424


In [10]:
# Break 2: fill missing prices with zero, then recompute average order value by category
zero = df.copy()
zero['unit_price'] = zero['unit_price'].fillna(0)
zero['order_value'] = zero['quantity'] * zero['unit_price']
comparison = pd.DataFrame({'zero_filled': zero.groupby('category')['order_value'].mean(),
                           'my_cleaned': df_clean.groupby('category')['order_value'].mean()})
comparison['difference'] = comparison['zero_filled'] - comparison['my_cleaned']
print(comparison.round(2))

             zero_filled  my_cleaned  difference
category                                        
Apparel           812.02      873.14      -61.12
Electronics       853.49      918.43      -64.94
Grocery           910.04      929.40      -19.36
Home             1006.92     1037.13      -30.21
Toys             1045.05     1065.95      -20.90


In [11]:
# Break 3: convert order_date with errors='raise'
converted = pd.to_datetime(df['order_date'], errors='raise')
print(converted.dtype, converted.min(), converted.max())

datetime64[us] 2026-01-01 00:00:00 2026-10-28 00:00:00


In [12]:
df_clean.to_csv('m4_orders_cleaned.csv', index=False)
print('saved', df_clean.shape)

saved (500, 10)
